In [ ]:
import os
import sys
import pandas as pd

# ==============================================================================
# 1. DYNAMIC DIRECTORY ALIGNMENT
# ==============================================================================
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))

# Inject project root into Python path so it can find 'src'
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"📁 Notebook directory: {notebook_dir}")
print(f"🚀 Project Root: {project_root}")

# Prevent pandas truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ==============================================================================
# 2. IMPORT MODULES FROM SRC
# ==============================================================================
try:
    from src.data_loader import load_raw_csv, clean_conv_pct
    from src.strikers import find_breakout_strikers
    print("✅ src modules successfully imported!")
except ModuleNotFoundError as e:
    print(f"❌ IMPORT ERROR: {e}")

# ==============================================================================
# 3. ROUTED DATA LOADING (Navigating data -> raw -> file)
# ==============================================================================
filename = "all_players_y1.csv"

# Added "raw" to step directly into your subfolder
csv_filename = os.path.join(project_root, "data", "raw", filename)

if os.path.exists(csv_filename):
    print(f"📊 Found raw data file at: {csv_filename}")
    
    raw_df = load_raw_csv(csv_filename)
    cleaned_df = clean_conv_pct(raw_df)
    
    # Run the TOPSIS Engine
    print("🤖 Processing Multi-Criteria TOPSIS Optimization...")
    striker_report = find_breakout_strikers(cleaned_df, min_minutes=700, max_minutes=2000)
    
    # Save the output file one level up in 'data' to keep 'raw' pristine
    output_path = os.path.join(project_root, "data", "global_breakout_strikers.csv")
    striker_report.to_csv(output_path, index=False)
    print(f"💾 Optimized report saved to: {output_path}")
    
    # Display results
    display(striker_report)
else:
    print(f"❌ DATA FILE NOT FOUND at: {csv_filename}")
    print(f"Please double check your file is inside: {os.path.join(project_root, 'data', 'raw')}")

In [ ]:
import os
import sys
import pandas as pd

# ==============================================================================
# 1. PATH SETUP & PACKAGING CHECKS
# ==============================================================================
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import load_raw_csv, clean_conv_pct
from src.strikers import find_breakout_strikers

# ==============================================================================
# 2. DEFINE AND LOAD HISTORICAL CSV FILES
# ==============================================================================
season1_filename = "all_players_y1.csv"  
season2_filename = "all_players_y2.csv"  

s1_path = os.path.join(project_root, "data", "raw", season1_filename)
s2_path = os.path.join(project_root, "data", "raw", season2_filename)

if not os.path.exists(s1_path) or not os.path.exists(s2_path):
    print("❌ ERROR: One or both historical files could not be found.")
else:
    print("📊 Loading and preparing historical data...")
    season1_df = clean_conv_pct(load_raw_csv(s1_path))
    season2_df = clean_conv_pct(load_raw_csv(s2_path))
    
    print(f"✅ Data processed! S1 ({len(season1_df)} players) | S2 ({len(season2_df)} players)")

    # ==============================================================================
    # 3. GENERATE SCORES FOR BOTH SEASONS
    # ==============================================================================
    print("🔮 Calculating Season 1 TOPSIS Index Scores...")
    s1_predictions = find_breakout_strikers(season1_df, min_minutes=700, max_minutes=2000)
    s1_scores = s1_predictions[['Player', 'Scouting_Index_Score']].copy()
    s1_scores.rename(columns={'Scouting_Index_Score': 'S1_Index_Score'}, inplace=True)

    print("🔮 Calculating Season 2 TOPSIS Index Scores...")
    # We open up max_minutes to 99999 so full-season starters aren't filtered out of the S2 ranking math
    s2_predictions = find_breakout_strikers(season2_df, min_minutes=700, max_minutes=99999)
    s2_scores = s2_predictions[['Player', 'Scouting_Index_Score']].copy()
    s2_scores.rename(columns={'Scouting_Index_Score': 'S2_Index_Score'}, inplace=True)

    # ==============================================================================
    # 4. GRAB RAW S2 EVOLUTION & MERGE EVERYTHING
    # ==============================================================================
    s2_natural_evolution = season2_df[season2_df['Minutes'] >= 900][['Player', 'Club', 'Minutes', 'Goals per 90 minutes', 'xG/90']].copy()
    s2_natural_evolution.rename(columns={
        'Minutes': 'S2_Minutes',
        'Goals per 90 minutes': 'S2_Actual_G90',
        'xG/90': 'S2_Actual_xG90'
    }, inplace=True)

    # Merge Season 1 predictions with their Season 2 playing stats
    validation_df = pd.merge(s1_scores, s2_natural_evolution, on='Player', how='inner')
    
    # Left-join the Season 2 algorithm scores so we can compare the evaluation shift
    validation_df = pd.merge(validation_df, s2_scores, on='Player', how='left')

    # Reorder columns logically to put the two scouting index scores right next to each other
    column_order = [
        'Player', 'S1_Index_Score', 'S2_Index_Score', 'Club', 
        'S2_Minutes', 'S2_Actual_G90', 'S2_Actual_xG90'
    ]
    validation_df = validation_df[column_order]
    validation_df = validation_df.sort_values(by='S1_Index_Score', ascending=False).reset_index(drop=True)

    # ==============================================================================
    # 5. REPORT CARD OUTPUT
    # ==============================================================================
    if len(validation_df) < 5:
        print("⚠️ The test pool is too small to calculate reliable stats.")
    else:
        correlation = validation_df['S1_Index_Score'].corr(validation_df['S2_Actual_G90'])
        
        print("\n" + "="*75)
        print("📊 NATURAL EVOLUTION ACCURACY REPORT (WITH SEASON-OVER-SEASON SCORES)")
        print("="*75)
        print(f"👥 Total low-minute targets AI naturally utilized in S2: {len(validation_df)}")
        print(f"📈 S1 Model Score to Future Goal Correlation Rate: {correlation:.2f}")
        print("="*75 + "\n")
        
        print("📋 Top 50 Predictions Tracking Sheet:")
        pd.set_option('display.max_rows', 55)
        display(validation_df.head(50))

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==============================================================================
# 0. HELPER: CLEAN CONV % (handles "23.4%" strings or already-numeric columns)
# ==============================================================================
def clean_conv_pct_numeric(series, fallback=12.0):
    if series.dtype == object:
        return pd.to_numeric(
            series.astype(str).str.replace('%', '', regex=False),
            errors='coerce'
        ).fillna(fallback)
    return series.fillna(fallback)


# ==============================================================================
# 1. FEATURE ENGINEERING BASED ON YOUR EXACT SCHEMA
# ==============================================================================
print("\U0001F6E0\uFE0F Engineering normalized performance features from raw columns...")

def prepare_ml_features(df, season_label, league_label, min_minutes=450):
    df_copy = df.copy()

    # Guard against division-by-zero / tiny-sample noise in per-90 stats
    df_copy = df_copy[df_copy['Minutes'] >= min_minutes].copy()

    # Standardize raw volumes into rate-metrics per 90 minutes
    df_copy['Shots_per_90'] = (df_copy['Shots'] / df_copy['Minutes']) * 90
    df_copy['ShT_per_90'] = (df_copy['ShT'] / df_copy['Minutes']) * 90

    # Clean Conv % so it's numeric before it ever reaches the model
    df_copy['Conv %'] = clean_conv_pct_numeric(df_copy['Conv %'])

    # Tag which league this row came from (used as a one-hot feature later)
    df_copy['League'] = league_label

    keep_cols = {
        'Player': 'Player',
        'League': 'League',
        'Minutes': f'{season_label}_Minutes',
        'Conv %': f'{season_label}_Conv_Pct',
        'Goals per 90 minutes': f'{season_label}_Goals_per_90',
        'xG/90': f'{season_label}_xG_per_90',
        'Drb/90': f'{season_label}_Drb_per_90',
        'Shots_per_90': f'{season_label}_Shots_per_90',
        'ShT_per_90': f'{season_label}_ShT_per_90'
    }

    return df_copy[list(keep_cols.keys())].rename(columns=keep_cols)


def build_league_ml_df(season1_df, season2_df, league_label):
    """Build the merged S1-features -> S2-target dataframe for one league."""
    s1_feat = prepare_ml_features(season1_df, 'S1', league_label)

    s2_stable = season2_df[season2_df['Minutes'] >= 900].copy()
    s2_target = s2_stable[['Player', 'Goals per 90 minutes']].rename(
        columns={'Goals per 90 minutes': 'Target_S2_G90'}
    )

    merged = pd.merge(s1_feat, s2_target, on='Player', how='inner').dropna()
    return merged


# ==============================================================================
# 2. LOAD WOMEN'S LEAGUE DATA (already loaded as season1_df / season2_df above)
#    + MEN'S LEAGUE DATA (new CSVs)
# ==============================================================================
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import load_raw_csv, clean_conv_pct

# --- File locations -----------------------------------------------------
# NOTE: double check "all_mplayers_y2.csv" for the women's S2 file below —
# if that's actually meant to be "all_wplayers_y2.csv", fix it here.
league_files = {
    'Womens': ("all_wplayers_y1.csv", "all_mplayers_y2.csv"),
    'Mens':   ("all_mplayers_y1.csv", "all_mplayers_y2.csv"),
}

raw_dir = os.path.join(project_root, "data", "raw")

frames = []
for league_label, (s1_filename, s2_filename) in league_files.items():
    s1_path = os.path.join(raw_dir, s1_filename)
    s2_path = os.path.join(raw_dir, s2_filename)

    if os.path.exists(s1_path) and os.path.exists(s2_path):
        s1_df = clean_conv_pct(load_raw_csv(s1_path))
        s2_df = clean_conv_pct(load_raw_csv(s2_path))

        league_ml_df = build_league_ml_df(s1_df, s2_df, league_label=league_label)
        frames.append(league_ml_df)
        print(f"\u2705 {league_label} league rows: {len(league_ml_df)}")
    else:
        print(f"\u26A0\uFE0F {league_label} CSVs not found at expected paths:\n   {s1_path}\n   {s2_path}")

if not frames:
    raise FileNotFoundError("No league data files were found in data/raw/. Check filenames above.")

# Combine leagues
ml_df_raw = pd.concat(frames, ignore_index=True)

# One-hot encode the League feature so the model can learn league-specific baselines
ml_df = pd.get_dummies(ml_df_raw, columns=['League'], drop_first=True)

ml_feature_names = [c for c in ml_df.columns if c not in ('Player', 'Target_S2_G90')]

X = ml_df[ml_feature_names]
y = ml_df['Target_S2_G90']

print(f"\u2705 Matrix built. Mapped {X.shape[0]} historical player records across {len(frames)} league(s).")

if X.shape[0] < 30:
    print(
        f"\u26A0\uFE0F  WARNING: Only {X.shape[0]} rows in the training set. "
        "With this few samples, R\u00b2/RMSE and feature importances will be "
        "high-variance and should be treated as directional, not definitive."
    )

# ==============================================================================
# 3. MODEL TRAINING & VALIDATION
# ==============================================================================

# --- 3a. Cross-validated estimate (more stable than a single split on small data) ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)

cv_r2 = cross_val_score(cv_model, X, y, cv=cv, scoring='r2')
cv_rmse = -cross_val_score(cv_model, X, y, cv=cv, scoring='neg_root_mean_squared_error')

print("\n" + "=" * 50)
print("\U0001F916 CROSS-VALIDATED ENGINE REPORT (5-fold)")
print("=" * 50)
print(f"\U0001F4C8 R\u00b2 across folds:   {np.round(cv_r2, 2)}")
print(f"   Mean R\u00b2:           {cv_r2.mean():.2f}  (\u00b1 {cv_r2.std():.2f})")
print(f"\U0001F4C9 RMSE across folds: {np.round(cv_rmse, 3)}")
print(f"   Mean RMSE:         {cv_rmse.mean():.3f} Goals/90 (\u00b1 {cv_rmse.std():.3f})")
print("=" * 50 + "\n")

# --- 3b. Single train/test split, kept for the importance plot below ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("\U0001F916 SINGLE HOLD-OUT REPORT (for reference only)")
print("=" * 50)
print(f"\U0001F4C9 Model Error (RMSE): {rmse:.3f} Goals/90 variance")
print(f"\U0001F4C8 Variance Explained (R\u00b2 Score): {r2:.2f}")
print("=" * 50 + "\n")

# ==============================================================================
# 4. EXTRACTION OF MATHEMATICAL PATTERNS
# ==============================================================================
final_model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
final_model.fit(X, y)

importances = final_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\U0001F4CB RANKED GOALSCORER PATTERNS (By Error Reduction Weight):")
print("This breaks down exactly which Season 1 features (and league) carry the highest predictive signal.")
print("-" * 75)
for f in range(X.shape[1]):
    print(f"{f + 1}. Feature: {X.columns[indices[f]]:<25} | Weight Impact: {importances[indices[f]]*100:.2f}%")
print("-" * 75)
print(
    "Note: Random Forest importances split credit unevenly among correlated "
    "features (e.g. Goals/90, xG/90, Shots/90 tend to move together). Treat "
    "the ranking as 'this cluster of related stats matters', not a precise "
    "ranking of independent causes. The League_Mens/League_Womens flag (if "
    "present) tells you whether the model thinks league context shifts the "
    "baseline goal rate on its own."
)

# Output pattern visualization plot
plt.figure(figsize=(10, 5))
plt.title("Random Forest Feature Importance - Striker Performance Patterns (All Leagues)")
plt.bar(range(X.shape[1]), importances[indices], align="center", color='darkslateblue')
plt.xticks(range(X.shape[1]), [X.columns[i] for i in indices], rotation=35, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
print("📋 Your actual dataset columns are:\n", season1_df.columns.tolist())

In [4]:
import os
import sys
import pandas as pd

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import load_raw_csv, clean_conv_pct
from src.scouting_engine import train_future_g90_model, find_active_breakouts

# ==============================================================================
# 1. LOAD HISTORICAL Y1 -> Y2 PAIRS FOR BOTH LEAGUES (for training the ML model)
# ==============================================================================
raw_dir = os.path.join(project_root, "data", "raw")

league_files = {
    'Womens': ("all_wplayers_y1.csv", "all_wplayers_y2.csv"),  # double-check y2 filename
    'Mens':   ("all_mplayers_y1.csv", "all_mplayers_y2.csv"),
}

league_history = {}
for league_label, (s1_name, s2_name) in league_files.items():
    s1_path = os.path.join(raw_dir, s1_name)
    s2_path = os.path.join(raw_dir, s2_name)
    league_history[league_label] = (
        clean_conv_pct(load_raw_csv(s1_path)),
        clean_conv_pct(load_raw_csv(s2_path)),
    )

# ==============================================================================
# 2. TRAIN THE FUTURE-G90 PREDICTOR ON HISTORICAL TRANSITIONS
# ==============================================================================
model, feature_cols, n_train = train_future_g90_model(league_history)
print(f"\u2705 Trained on {n_train} historical player transitions across {len(league_history)} leagues.")
print(f"   Feature columns: {feature_cols}")

# ==============================================================================
# 3. SCOUT THE CURRENT (MOST RECENT) SEASON FOR ACTIVE BREAKOUTS
# ==============================================================================
# Use each league's most recent ("y2") season as the "current" data to scout from.
womens_breakouts = find_active_breakouts(
    league_history['Womens'][1], league_label='Womens',
    model=model, feature_cols=feature_cols,
    min_minutes=450, max_minutes=2000, max_age=24, min_uplift=0.05, top_n=25,
)

mens_breakouts = find_active_breakouts(
    league_history['Mens'][1], league_label='Mens',
    model=model, feature_cols=feature_cols,
    min_minutes=450, max_minutes=2000, max_age=24, min_uplift=0.05, top_n=25,
)

print("\n" + "=" * 75)
print("\U0001F50D ACTIVE BREAKOUT CANDIDATES \u2014 WOMEN'S LEAGUE")
print("=" * 75)
display(womens_breakouts)

print("\n" + "=" * 75)
print("\U0001F50D ACTIVE BREAKOUT CANDIDATES \u2014 MEN'S LEAGUE")
print("=" * 75)
display(mens_breakouts)


✅ Trained on 2155 historical player transitions across 2 leagues.
   Feature columns: ['Minutes', 'Conv %', 'Goals per 90 minutes', 'xG/90', 'Drb/90', 'Shots_per_90', 'ShT_per_90', 'League_Mens', 'League_Womens']

🔍 ACTIVE BREAKOUT CANDIDATES — WOMEN'S LEAGUE


,Player,Club,Age,Minutes,Goals per 90 minutes,Predicted_Future_G90,G90_Uplift,xG/90,Conv %,Scouting_Index_Score
0,Soufiya Nina Ngueleu,Montpellier,22.0,747.0,0.0,0.377693,0.377693,0.5,0.0,30.380272
1,Iris Santiago,R. Madrid B,19.0,556.0,0.0,0.356918,0.356918,0.4,0.0,NaN
2,Estefanía González,Europa,21.0,774.0,0.6,0.926007,0.326007,0.9,9.0,52.551658
3,Fortesa Berisha,Como Women,23.0,466.0,0.0,0.311557,0.311557,0.3,0.0,NaN
4,Sigdís Bárðardóttir,IFK Norrköping,20.0,545.0,0.0,0.287599,0.287599,0.1,0.0,NaN
5,Mihiro Moteki,RB Omiya Ardija,19.0,476.0,0.0,0.286740,0.286740,0.5,0.0,NaN
6,Antía Mayo,S. Gijón,20.0,484.0,0.0,0.284224,0.284224,0.3,0.0,NaN
7,Izabella Rako,Melbourne City,17.0,566.0,0.2,0.475704,0.275704,0.7,3.0,NaN
8,Carolina Santiago,FC Bayern München,20.0,1591.0,0.5,0.761223,0.261223,0.6,8.0,52.217689
9,María Ólafsdóttir Grós,Djurgårdens IF,24.0,797.0,0.1,0.351796,0.251796,0.3,3.0,NaN



🔍 ACTIVE BREAKOUT CANDIDATES — MEN'S LEAGUE


,Player,Club,Age,Minutes,Goals per 90 minutes,Predicted_Future_G90,G90_Uplift,xG/90,Conv %,Scouting_Index_Score
0,Conrad Harder,RB Leipzig,22,1145,0.5,1.044606,0.544606,0.6,8,52.920607
1,Franjo Ivanović,Benfica,23,508,0.4,0.880400,0.480400,0.7,5,NaN
2,Francesco Pio Esposito,Blu-neri,21,1757,0.6,0.986678,0.386678,0.7,10,67.248714
3,Antonio Arena,AS Roma,18,486,0.0,0.337340,0.337340,0.3,0,NaN
4,Pape Moussa Fall,FC Metz,22,475,0.2,0.519032,0.319032,1.1,4,NaN
5,Franco Mastantuono,R. Madrid,19,879,0.1,0.399431,0.299431,0.4,3,NaN
6,Valerio Cinti,AS Roma,19,497,0.0,0.298795,0.298795,0.2,0,NaN
7,Alejandro Gomes Rodríguez,OL,19,803,0.0,0.285019,0.285019,0.3,0,27.745698
8,Ryan Metu,FC Groningen,18,485,0.0,0.277212,0.277212,0.3,0,NaN
9,Otto Stange,Hamburger SV,20,604,0.0,0.264192,0.264192,0.3,0,NaN


In [ ]:
import os
import sys
import pandas as pd

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import load_raw_csv, clean_conv_pct
from src.scouting_engine import predict_future_g90, find_active_breakouts, FEATURE_COLS

# ==============================================================================
# SCOUT FRESH DATA WITH THE TRAINED MODEL
# ==============================================================================
# Point this at whatever new/current CSV(s) you want to scout. These should
# have the SAME raw columns as your historical files (Unique ID, Player,
# Club, Best Pos, Age, Minutes, Goals, xG, Shots, ShT, Conv %,
# Goals per 90 minutes, xG/90, Drb/90, etc.) but can be any snapshot in time
# — e.g. mid-season form, a new league, a lower-division feed, etc.

fresh_files = {
    'Womens': "all_wplayers_fresh.csv",
    'Mens':   "all_mplayers_fresh.csv",
}

raw_dir = os.path.join(project_root, "data", "raw")

fresh_breakouts = {}

for league_label, filename in fresh_files.items():
    path = os.path.join(raw_dir, filename)

    if not os.path.exists(path):
        print(f"\u26A0\uFE0F Skipping {league_label}: file not found at {path}")
        continue

    fresh_df = clean_conv_pct(load_raw_csv(path))

    # --- Safe feature alignment check ------------------------------------
    # Make sure every column the model was trained on (besides the
    # League_* dummies, which predict_future_g90 fills in itself) is
    # actually present in this fresh dataset before we run predictions.
    missing_cols = [c for c in FEATURE_COLS if c not in fresh_df.columns]
    if missing_cols:
        print(f"\u26A0\uFE0F Skipping {league_label}: missing required columns {missing_cols}")
        continue

    breakouts = find_active_breakouts(
        fresh_df, league_label=league_label,
        model=model, feature_cols=feature_cols,
        min_minutes=450, max_minutes=2000, max_age=24, min_uplift=0.05, top_n=25,
    )

    breakouts.insert(0, 'League', league_label)
    fresh_breakouts[league_label] = breakouts

    print(f"\u2705 {league_label}: {len(breakouts)} breakout candidates found "
          f"from {len(fresh_df)} players.")

# ==============================================================================
# CLEAN, RANKED, COMBINED TARGET LIST
# ==============================================================================
if fresh_breakouts:
    combined = pd.concat(fresh_breakouts.values(), ignore_index=True)
    combined = combined.sort_values(by='G90_Uplift', ascending=False).reset_index(drop=True)

    # Round numeric columns for a cleaner display
    for col in ['Goals per 90 minutes', 'Predicted_Future_G90', 'G90_Uplift', 'xG/90', 'Scouting_Index_Score']:
        if col in combined.columns:
            combined[col] = combined[col].round(2)

    print("\n" + "=" * 75)
    print("\U0001F3AF FRESH SCOUTING TARGET LIST \u2014 RANKED BY PROJECTED UPLIFT")
    print("=" * 75)
    display(combined)

    # Save for downstream use
    output_path = os.path.join(project_root, "data", "active_breakout_targets.csv")
    combined.to_csv(output_path, index=False)
    print(f"\n\U0001F4BE Saved target list to: {output_path}")
else:
    print("\u26A0\uFE0F No fresh data files found \u2014 update the filenames in fresh_files above.")
